# Validación de la meteorología ERA5-Land

Este notebook comprueba el cubo meteorológico diario 2018-12-01 a 2023-12-31 y visualiza una variable sin generar archivos PNG.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white"})

DAILY_PATH = Path("../data/processed/meteorology/era5_daily_2018_2023.nc")
CUBE_PATH = Path("../data/processed/meteorology_2018_2023.nc")
FECHA = "2022-07-15"
VARIABLE = "temperature_max_12_18h"

In [ ]:
daily = xr.open_dataset(DAILY_PATH)
meteo = xr.open_dataset(CUBE_PATH)

print("Periodo diario:", str(daily.time.min().values)[:10], "a", str(daily.time.max().values)[:10])
print("Periodo del cubo:", str(meteo.time.min().values)[:10], "a", str(meteo.time.max().values)[:10])
print("Variables:", list(daily.data_vars))
print("Dimensiones del cubo:", dict(meteo.sizes))

In [ ]:
active = meteo["is_galicia"] == 1
valid = meteo[VARIABLE].notnull() & active

print(f"Celdas activas: {int(active.sum())}")
print(f"Valores válidos de {VARIABLE}: {int(valid.sum())}")
print(f"Cobertura sobre celdas activas: {float(valid.sum() / active.sum() / meteo.sizes['time'] * 100):.2f}%")
assert str(meteo.time.min().values)[:10] == "2018-12-01"
assert str(meteo.time.max().values)[:10] == "2023-12-31"

In [ ]:
units = meteo[VARIABLE].attrs.get("units", "")
layer = meteo[VARIABLE].sel(time=FECHA).where(active)

fig, ax = plt.subplots(figsize=(9, 8), facecolor="white")
layer.plot(ax=ax, cmap="inferno", cbar_kwargs={"label": f"{VARIABLE} ({units})"})
ax.set_title(f"{VARIABLE} — {FECHA}")
ax.set_xlabel("x (EPSG:3035)")
ax.set_ylabel("y (EPSG:3035)")
ax.set_facecolor("white")
plt.show()

In [ ]:
daily.close()
meteo.close()